## 2026 EY AI & Data Challenge - TerraClimate Data Extraction Notebook

This notebooks demonstrates how to access the TerraClimate dataset. TerraClimate is a dataset of monthly climate and climatic water balance for global terrestrial surfaces from 1958 to the present. These data provide important inputs for ecological and hydrological studies at global scales that require high spatial resolution and time-varying data. All data have monthly temporal resolution and a ~4-km (1/24th degree) spatial resolution. This dataset is provided in Zarr format. 

For more information, visit: https://planetarycomputer.microsoft.com/dataset/terraclimate#overview 

In [51]:
import warnings
warnings.filterwarnings("ignore")

# Data manipulation and analysis
import numpy as np
import pandas as pd

# Multi-dimensional arrays and datasets (e.g., NetCDF, Zarr)
import xarray as xr

from scipy.spatial import cKDTree

# Planetary Computer tools for STAC API access and authentication
import pystac_client
import planetary_computer as pc

from datetime import date
from tqdm import tqdm
import os
import time

import os, certifi
os.environ["SSL_CERT_FILE"] = certifi.where()

# Chunked/resumable config - save every 60 samples to avoid API timeouts
CHUNK_SIZE = 10  # Process and save every 60 samples
RESUME = True
CHUNK_RETRIES = 3
RETRY_SLEEP_S = 5
SAVE_EVERY_ROWS = 10

# All extraction outputs go to tc_filtered (checkpoints + final CSVs)
OUTPUT_DIR = os.path.join(os.getcwd(), "tc_filtered")
os.makedirs(OUTPUT_DIR, exist_ok=True)
FILTER_CACHE_DIR = OUTPUT_DIR  # Filtered grid cache
CHECKPOINT_DIR = OUTPUT_DIR    # Per-variable extraction checkpoints


<h2>Extracting TerraClimate Data Using API Calls</h2> <p align="justify"> The API-based method allows us to efficiently access <b>TerraClimate</b> data for specific regions and time periods through the <a href="https://planetarycomputer.microsoft.com/">Microsoft Planetary Computer</a>, ensuring scalability and reproducibility of the process. </p> <p align="justify"> Through the API, we can extract climate variables such as <b>Potential Evapotranspiration (PET)</b>, which represents the atmospheric demand for water. This variable provides critical insights into surface moisture balance and helps improve the accuracy of water quality modeling. </p> <p align="justify"> This approach ensures consistent, automated retrieval of high-resolution climate data that can be easily integrated with satellite-derived features for comprehensive environmental and hydrological analysis. </p>



<h3>Loading and Mapping TerraClimate Data:</h3>

<p>This section demonstrates how <b>TerraClimate climate variables</b>, such as <b>Potential Evapotranspiration (PET)</b>, are loaded and mapped to sampling locations:</p>

<ul>
  <li>The <b>load_terraclimate_dataset</b> function opens the TerraClimate Zarr/NetCDF dataset from the Microsoft Planetary Computer, handling storage options automatically.</li>
  <li>The <b>filterg</b> function filters the dataset for the desired time range (2011–2015) and spatial extent corresponding to the study region. The resulting data is converted to a pandas DataFrame with standardized column names.</li>
  <li>The <b>assign_nearest_climate</b> function maps each sampling location to its <b>nearest TerraClimate grid point</b> using a KD-tree and assigns the climate variable values corresponding to the closest time stamp.</li>
</ul>

<p>This workflow ensures efficient, reproducible retrieval of climate variables, while allowing participants to work with pre-extracted CSV files for faster benchmarking and analysis.</p>


In [52]:
def load_terraclimate_dataset():
    catalog = pystac_client.Client.open(
        "https://planetarycomputer.microsoft.com/api/stac/v1",
        modifier=pc.sign_inplace,
    )
    collection = catalog.get_collection("terraclimate")
    asset = collection.assets["zarr-abfs"]

    if "xarray:storage_options" in asset.extra_fields:
        ds = xr.open_zarr(
            asset.href,
            storage_options=asset.extra_fields["xarray:storage_options"],
            consolidated=True,
        )
    else:
        ds = xr.open_dataset(
            asset.href,
            **asset.extra_fields["xarray:open_kwargs"],
        )

    return ds

In [53]:
# Override filterg with on-disk cache

def filterg(ds, var):
    cache_path = os.path.join(FILTER_CACHE_DIR, f"{var}_filtered.csv")
    if os.path.exists(cache_path):
        return pd.read_csv(cache_path)

    ds_2011_2015 = ds[var].sel(time=slice("2011-01-01", "2015-12-31"))

    df_var_append = []
    for i in tqdm(range(len(ds_2011_2015.time))):
        df_var = ds_2011_2015.isel(time=i).to_dataframe().reset_index()
        df_var_filter = df_var[
            (df_var['lat'] > -35.18) & (df_var['lat'] < -21.72) &
            (df_var['lon'] > 14.97) & (df_var['lon'] < 32.79)
        ]
        df_var_append.append(df_var_filter)

    df_var_final = pd.concat(df_var_append, ignore_index=True)
    print(f"Filtering for {var} completed")

    df_var_final['time'] = df_var_final['time'].astype(str)

    # Column mapping
    col_mapping = {"lat": "Latitude", "lon": "Longitude", "time": "Sample Date"}
    df_var_final = df_var_final.rename(columns=col_mapping)

    df_var_final.to_csv(cache_path, index=False)
    return df_var_final


In [54]:
# Override assign_nearest_climate_chunked to resume within a chunk

def assign_nearest_climate_chunked(sa_df, climate_df, var_name, checkpoint_path):
    """Row-level checkpointing to resume mid-chunk."""
    sa_df = sa_df.reset_index(drop=True)

    # Precompute nearest grid point for each sample
    sa_coords = np.radians(sa_df[['Latitude', 'Longitude']].values)
    climate_coords = np.radians(climate_df[['Latitude', 'Longitude']].values)
    tree = cKDTree(climate_coords)
    dist, idx = tree.query(sa_coords, k=1)
    nearest_points = climate_df.iloc[idx].reset_index(drop=True)
    sa_df[['nearest_lat', 'nearest_lon']] = nearest_points[['Latitude', 'Longitude']]

    sa_df['Sample Date'] = pd.to_datetime(sa_df['Sample Date'], dayfirst=True, errors='coerce')
    climate_df['Sample Date'] = pd.to_datetime(climate_df['Sample Date'], dayfirst=True, errors='coerce')

    start_idx = 0
    if RESUME and os.path.exists(checkpoint_path):
        existing = pd.read_csv(checkpoint_path)
        start_idx = len(existing)
        print(f"Resuming {var_name} from row {start_idx}")

    total = len(sa_df)
    buffer = []

    def _flush_buffer():
        nonlocal buffer
        if not buffer:
            return
        df_out = pd.DataFrame({var_name: buffer})
        write_header = not os.path.exists(checkpoint_path) or start_idx == 0 and os.path.getsize(checkpoint_path) == 0
        df_out.to_csv(checkpoint_path, mode='a', header=write_header, index=False)
        buffer = []

    try:
        for i in range(start_idx, total):
            sample_date = sa_df.loc[i, 'Sample Date']
            nearest_lat = sa_df.loc[i, 'nearest_lat']
            nearest_lon = sa_df.loc[i, 'nearest_lon']

            subset = climate_df[
                (climate_df['Latitude'] == nearest_lat) &
                (climate_df['Longitude'] == nearest_lon)
            ]

            if subset.empty:
                buffer.append(np.nan)
            else:
                nearest_idx = (subset['Sample Date'] - sample_date).abs().idxmin()
                buffer.append(subset.loc[nearest_idx, var_name])

            # Save every N rows
            if (i + 1) % SAVE_EVERY_ROWS == 0:
                _flush_buffer()

        _flush_buffer()
    except Exception as e:
        # Save progress before raising
        _flush_buffer()
        raise

    return pd.read_csv(checkpoint_path)


In [55]:
# Incremental + cached TerraClimate filter
# Writes filtered grids per variable into tc_filtered and resumes
# across runs so long variables like "ppt" don't restart from 0.
TIME_SAVE_EVERY = 60  # save progress every 60 monthly slices


def filterg(ds, var):
    base_name = f"{var}_filtered"
    cache_path = os.path.join(FILTER_CACHE_DIR, base_name + ".csv")
    progress_path = os.path.join(FILTER_CACHE_DIR, base_name + "_progress.txt")

    # If we already have a completed cache and no progress marker, just load it
    if os.path.exists(cache_path) and not os.path.exists(progress_path):
        return pd.read_csv(cache_path)

    ds_2011_2015 = ds[var].sel(time=slice("2011-01-01", "2011-12-31"))
    ds_2011_2015 = ds[var].sel(time=slice("2011-01-01", "2015-12-31"))
    n_times = len(ds_2011_2015.time)

    # Resume from last completed time index if a progress file exists
    start_i = 0
    if os.path.exists(progress_path):
        try:
            with open(progress_path, "r", encoding="utf-8") as f:
                start_i = int(f.read().strip() or "0")
        except Exception:
            start_i = 0

    write_header = not os.path.exists(cache_path) or start_i == 0

    print(f"Filtering for {var} from time index {start_i} of {n_times}...")

    for i in range(start_i, n_times):
        # Single time slice -> DataFrame
        df_var = ds_2011_2015.isel(time=i).to_dataframe().reset_index()
        df_var_filter = df_var[
            (df_var["lat"] > -35.18) & (df_var["lat"] < -21.72) &
            (df_var["lon"] > 14.97) & (df_var["lon"] < 32.79)
        ]

        if not df_var_filter.empty:
            # Standardize columns and write incrementally
            df_var_filter["time"] = df_var_filter["time"].astype(str)
            col_mapping = {"lat": "Latitude", "lon": "Longitude", "time": "Sample Date"}
            df_var_filter = df_var_filter.rename(columns=col_mapping)

            df_var_filter.to_csv(
                cache_path,
                mode="a" if not write_header else "w",
                header=write_header,
                index=False,
            )
            write_header = False

        # Every TIME_SAVE_EVERY steps (or at the end), update progress and pause briefly
        if (i + 1) % TIME_SAVE_EVERY == 0 or i == n_times - 1:
            with open(progress_path, "w", encoding="utf-8") as f:
                f.write(str(i + 1))
            print(f"{var}: finished time index {i} (wrote to {cache_path})")
            time.sleep(1)

    # Completed this variable; remove progress marker and return full DataFrame
    if os.path.exists(progress_path):
        os.remove(progress_path)

    print(f"Filtering for {var} completed and cached at {cache_path}")
    return pd.read_csv(cache_path)


In [56]:
# --- Chunked + resumable mapping ---
def assign_nearest_climate_chunked(sa_df, climate_df, var_name, checkpoint_path):
    """Chunked + resumable mapping using checkpoint CSV."""
    sa_df = sa_df.reset_index(drop=True)

    # Precompute nearest grid point for each sample
    sa_coords = np.radians(sa_df[['Latitude', 'Longitude']].values)
    climate_coords = np.radians(climate_df[['Latitude', 'Longitude']].values)
    tree = cKDTree(climate_coords)
    dist, idx = tree.query(sa_coords, k=1)
    nearest_points = climate_df.iloc[idx].reset_index(drop=True)
    sa_df[['nearest_lat', 'nearest_lon']] = nearest_points[['Latitude', 'Longitude']]

    sa_df['Sample Date'] = pd.to_datetime(sa_df['Sample Date'], dayfirst=True, errors='coerce')
    climate_df['Sample Date'] = pd.to_datetime(climate_df['Sample Date'], dayfirst=True, errors='coerce')

    start_idx = 0
    if RESUME and os.path.exists(checkpoint_path):
        existing = pd.read_csv(checkpoint_path)
        start_idx = len(existing)
        print(f"Resuming {var_name} from row {start_idx}")
    else:
        existing = None

    results = []
    if existing is not None and start_idx > 0:
        results.append(existing)

    total = len(sa_df)
    for chunk_start in range(start_idx, total, CHUNK_SIZE):
        chunk_end = min(chunk_start + CHUNK_SIZE, total)

        for attempt in range(1, CHUNK_RETRIES + 1):
            try:
                chunk_vals = []
                for i in range(chunk_start, chunk_end):
                    sample_date = sa_df.loc[i, 'Sample Date']
                    nearest_lat = sa_df.loc[i, 'nearest_lat']
                    nearest_lon = sa_df.loc[i, 'nearest_lon']

                    subset = climate_df[
                        (climate_df['Latitude'] == nearest_lat) &
                        (climate_df['Longitude'] == nearest_lon)
                    ]

                    if subset.empty:
                        chunk_vals.append(np.nan)
                        continue

                    nearest_idx = (subset['Sample Date'] - sample_date).abs().idxmin()
                    chunk_vals.append(subset.loc[nearest_idx, var_name])

                chunk_df = pd.DataFrame({var_name: chunk_vals})
                results.append(chunk_df)

                # Save checkpoint every CHUNK_SIZE (60) samples to tc_filtered
                pd.concat(results, ignore_index=True).to_csv(checkpoint_path, index=False)
                print(f"{var_name}: saved rows {chunk_start}..{chunk_end - 1} to tc_filtered")
                time.sleep(1)  # Brief pause to reduce API timeout risk
                break
            except Exception as e:
                if attempt == CHUNK_RETRIES:
                    raise
                print(f"{var_name} chunk {chunk_start}-{chunk_end} failed (attempt {attempt}): {e}")
                time.sleep(RETRY_SLEEP_S * attempt)

    return pd.concat(results, ignore_index=True)


In [57]:
# --- Filtering function (kept identical) ---
def filterg(ds, var):
    ds_2011_2015 = ds[var].sel(time=slice("2011-01-01", "2015-12-31"))

    df_var_append = []
    for i in tqdm(range(len(ds_2011_2015.time))):
        df_var = ds_2011_2015.isel(time=i).to_dataframe().reset_index()
        df_var_filter = df_var[
            (df_var['lat'] > -35.18) & (df_var['lat'] < -21.72) &
            (df_var['lon'] > 14.97) & (df_var['lon'] < 32.79)
        ]
        df_var_append.append(df_var_filter)

    df_var_final = pd.concat(df_var_append, ignore_index=True)
    print(f"Filtering for {var} completed")

    df_var_final['time'] = df_var_final['time'].astype(str)

    # Column mapping
    col_mapping = {"lat": "Latitude", "lon": "Longitude", "time": "Sample Date"}
    df_var_final = df_var_final.rename(columns=col_mapping)

    return df_var_final


In [58]:
# --- Climate variable assignment function (unchanged logic) ---
def assign_nearest_climate(sa_df, climate_df, var_name):
    """
    Map nearest climate variable values to a new DataFrame 
    containing only the specified variable column.
    """
    sa_coords = np.radians(sa_df[['Latitude', 'Longitude']].values)
    climate_coords = np.radians(climate_df[['Latitude', 'Longitude']].values)

    tree = cKDTree(climate_coords)
    dist, idx = tree.query(sa_coords, k=1)

    nearest_points = climate_df.iloc[idx].reset_index(drop=True)

    sa_df = sa_df.reset_index(drop=True)
    sa_df[['nearest_lat', 'nearest_lon']] = nearest_points[['Latitude', 'Longitude']]

    sa_df['Sample Date'] = pd.to_datetime(sa_df['Sample Date'], dayfirst=True, errors='coerce')
    climate_df['Sample Date'] = pd.to_datetime(climate_df['Sample Date'], dayfirst=True, errors='coerce')

    climate_values = []

    for i in tqdm(range(len(sa_df)), desc=f"Mapping {var_name.upper()} values"):
        sample_date = sa_df.loc[i, 'Sample Date']
        nearest_lat = sa_df.loc[i, 'nearest_lat']
        nearest_lon = sa_df.loc[i, 'nearest_lon']

        subset = climate_df[
            (climate_df['Latitude'] == nearest_lat) &
            (climate_df['Longitude'] == nearest_lon)
        ]

        if subset.empty:
            climate_values.append(np.nan)
            continue

        nearest_idx = (subset['Sample Date'] - sample_date).abs().idxmin()
        climate_values.append(subset.loc[nearest_idx, var_name])

    output_df = pd.DataFrame({var_name: climate_values})

    
    return output_df

### Extracting features for the training dataset

In [59]:
# Load water quality training dataset (repo root)
Water_Quality_df = pd.read_csv('Datasets_Provided/water_quality_training_dataset.csv')
Water_Quality_df.head()

,Latitude,Longitude,Sample Date,Total Alkalinity,Electrical Conductance,Dissolved Reactive Phosphorus
0,-28.760833,17.730278,02-01-2011,128.912,555.0,10.0
1,-26.861111,28.884722,03-01-2011,74.720,162.9,163.0
2,-26.450000,28.085833,03-01-2011,89.254,573.0,80.0
3,-27.671111,27.236944,03-01-2011,82.000,203.6,101.0
4,-27.356667,27.286389,03-01-2011,56.100,145.1,151.0


In [60]:
Water_Quality_df.shape

(9319, 6)

In [61]:
# Load TerraClimate dataset, filter (time,region,parameter), filter for nearest parameter values
vars_to_extract = ["soil", "swe",
    "srad", "tmax", "tmin", "vap", "vpd", "ws", "pdsi",
]

ds = load_terraclimate_dataset()

feature_dfs = []
for var in vars_to_extract:
    while True:
        try:
            tc_parameter = filterg(ds, var)
            ckpt_path = os.path.join(CHECKPOINT_DIR, f"train_{var}.csv")
            feature_dfs.append(assign_nearest_climate_chunked(Water_Quality_df, tc_parameter, var, ckpt_path))
            break
        except Exception as e:
            err_text = str(e)
            # Refresh signed URL if token expires mid-run
            if "AuthenticationFailed" in err_text or "Signature not valid" in err_text:
                print(f"Token expired while processing {var}, refreshing dataset...")
                time.sleep(5)
                ds = load_terraclimate_dataset()
                continue
            # Retry on transient socket timeouts
            if "Timeout" in err_text or "SocketTimeoutError" in err_text or "ServiceResponseTimeoutError" in err_text:
                print(f"Timeout while processing {var}, retrying after backoff...")
                time.sleep(10)
                ds = load_terraclimate_dataset()
                continue
            raise

Terraclimate_training_df = pd.concat(feature_dfs, axis=1)

 52%|█████▏    | 31/60 [15:35<14:35, 30.18s/it]


Token expired while processing soil, refreshing dataset...


100%|██████████| 60/60 [34:40<00:00, 34.67s/it]


Filtering for soil completed
soil: saved rows 0..9 to tc_filtered
soil: saved rows 10..19 to tc_filtered
soil: saved rows 20..29 to tc_filtered
soil: saved rows 30..39 to tc_filtered
soil: saved rows 40..49 to tc_filtered
soil: saved rows 50..59 to tc_filtered
soil: saved rows 60..69 to tc_filtered
soil: saved rows 70..79 to tc_filtered
soil: saved rows 80..89 to tc_filtered
soil: saved rows 90..99 to tc_filtered
soil: saved rows 100..109 to tc_filtered
soil: saved rows 110..119 to tc_filtered
soil: saved rows 120..129 to tc_filtered
soil: saved rows 130..139 to tc_filtered
soil: saved rows 140..149 to tc_filtered
soil: saved rows 150..159 to tc_filtered
soil: saved rows 160..169 to tc_filtered
soil: saved rows 170..179 to tc_filtered
soil: saved rows 180..189 to tc_filtered
soil: saved rows 190..199 to tc_filtered
soil: saved rows 200..209 to tc_filtered
soil: saved rows 210..219 to tc_filtered
soil: saved rows 220..229 to tc_filtered
soil: saved rows 230..239 to tc_filtered
soil: sav

  0%|          | 0/60 [00:00<?, ?it/s]


Token expired while processing swe, refreshing dataset...


  0%|          | 0/60 [04:37<?, ?it/s]


Timeout while processing swe, retrying after backoff...


ServiceRequestError: Cannot connect to host cpdataeuwest.blob.core.windows.net:443 ssl:default [nodename nor servname provided, or not known]

In [62]:
Terraclimate_training_df['Latitude'] = Water_Quality_df['Latitude']
Terraclimate_training_df['Longitude'] = Water_Quality_df['Longitude']
Terraclimate_training_df['Sample Date'] = Water_Quality_df['Sample Date']

# Keep columns in requested order
Terraclimate_training_df = Terraclimate_training_df[
    ['Latitude', 'Longitude', 'Sample Date'] + vars_to_extract
]

# Save to tc_filtered folder
Terraclimate_training_df.to_csv(os.path.join(OUTPUT_DIR, 'terraclimate_training.csv'), index=False)

NameError: name 'Terraclimate_training_df' is not defined

In [ ]:
# Preview File
Terraclimate_training_df.head()

NameError: name 'Terraclimate_training_df' is not defined

### Extracting features for the validation dataset

In [71]:
# Load submission template (repo root)
Validation_df = pd.read_csv('submission_template.csv')
Validation_df.head()

,Latitude,Longitude,Sample Date,Total Alkalinity,Electrical Conductance,Dissolved Reactive Phosphorus
0,-32.043333,27.822778,01-09-2014,NaN,NaN,NaN
1,-33.329167,26.077500,16-09-2015,NaN,NaN,NaN
2,-32.991639,27.640028,07-05-2015,NaN,NaN,NaN
3,-34.096389,24.439167,07-02-2012,NaN,NaN,NaN
4,-32.000556,28.581667,01-10-2014,NaN,NaN,NaN


In [72]:
Validation_df.shape

(200, 6)

In [ ]:
### vars_to_extract = [
###    "pet", "aet", "def", "q", "ppt", "soil", "swe",
###    "srad", "tmax", "tmin", "vap", "vpd", "ws", "pdsi",
### ]

vars_to_extract = [
    "pet", "aet", "def", "q", "soil"
]

# Exact validation logic from TerraClimate_Data_Extraction_Notebook.ipynb (demo):
# filterg(ds, var) -> tc_parameter, assign_nearest_climate(Validation_df, tc_parameter, var)
try:
    ds
except NameError:
    ds = load_terraclimate_dataset()

feature_dfs = []
for var in vars_to_extract:
    while True:
        try:
            tc_parameter = filterg(ds, var)
            out_df = assign_nearest_climate(Validation_df, tc_parameter, var)
            feature_dfs.append(out_df)
            break
        except Exception as e:
            err_text = str(e)
            if "AuthenticationFailed" in err_text or "Signature not valid" in err_text or "ClientAuthenticationError" in err_text:
                print(f"Token expired during {var}, refreshing dataset...")
                time.sleep(5)
                ds = load_terraclimate_dataset()
                continue
            if "Timeout" in err_text or "SocketTimeoutError" in err_text or "ServiceResponseTimeoutError" in err_text:
                print(f"Timeout during {var}, retrying...")
                time.sleep(10)
                ds = load_terraclimate_dataset()
                continue
            raise

Terraclimate_validation_df = pd.concat(feature_dfs, axis=1)
Terraclimate_validation_df["Latitude"] = Validation_df["Latitude"].values
Terraclimate_validation_df["Longitude"] = Validation_df["Longitude"].values
Terraclimate_validation_df["Sample Date"] = Validation_df["Sample Date"].values
Terraclimate_validation_df = Terraclimate_validation_df[["Latitude", "Longitude", "Sample Date"] + vars_to_extract]
Terraclimate_validation_df.to_csv(os.path.join(OUTPUT_DIR, "tc_validation.csv"), index=False)

 72%|███████▏  | 43/60 [17:59<06:58, 24.65s/it]Incomplete download.
Incomplete download.
Incomplete download.
Incomplete download.
Incomplete download.
Incomplete download.
Incomplete download.


In [ ]:
Terraclimate_validation_df['Latitude'] = Validation_df['Latitude']
Terraclimate_validation_df['Longitude'] = Validation_df['Longitude']
Terraclimate_validation_df['Sample Date'] = Validation_df['Sample Date']

# Keep columns in requested order
Terraclimate_validation_df = Terraclimate_validation_df[
    ['Latitude', 'Longitude', 'Sample Date'] + vars_to_extract
]

# Save to tc_filtered folder
Terraclimate_validation_df.to_csv(os.path.join(OUTPUT_DIR, 'terraclimate_validation.csv'), index=False)

In [ ]:
# Preview File
Terraclimate_validation_df.head()

,Latitude,Longitude,Sample Date,pet
0,-32.043333,27.822778,01-09-2014,161.900009
1,-33.329167,26.077500,16-09-2015,177.600006
2,-32.991639,27.640028,07-05-2015,158.400009
3,-34.096389,24.439167,07-02-2012,130.000000
4,-32.000556,28.581667,01-10-2014,152.500000
